<a href="https://colab.research.google.com/github/umeshkumarsahu07/DMPM-LAB/blob/main/DMPM_EXP_06_DECISION_TREE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def gini(self, y):
        classes, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities ** 2)

    def best_split(self, X, y):
        best_gini = float("inf")
        best_feature = None
        best_threshold = None

        n_samples, n_features = X.shape

        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])

            for threshold in thresholds:
                left = X[:, feature] <= threshold
                right = ~left

                if np.sum(left) == 0 or np.sum(right) == 0:
                    continue

                y_left = y[left]
                y_right = y[right]

                weighted_gini = (
                    len(y_left) / n_samples * self.gini(y_left)
                    + len(y_right) / n_samples * self.gini(y_right)
                )

                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    def build_tree(self, X, y, depth):
        # Stop conditions
        if (
            depth >= self.max_depth
            or len(np.unique(y)) == 1
            or len(y) < self.min_samples_split
        ):
            return np.bincount(y).argmax()

        feature, threshold = self.best_split(X, y)

        if feature is None:
            return np.bincount(y).argmax()

        left = X[:, feature] <= threshold
        right = ~left

        return {
            "feature": feature,
            "threshold": threshold,
            "left": self.build_tree(X[left], y[left], depth + 1),
            "right": self.build_tree(X[right], y[right], depth + 1)
        }

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y, dtype=int)

        self.tree = self.build_tree(X, y, 0)
        return self

    def predict_one(self, x, node):
        if not isinstance(node, dict):
            return node

        if x[node["feature"]] <= node["threshold"]:
            return self.predict_one(x, node["left"])
        else:
            return self.predict_one(x, node["right"])

    def predict(self, X):
        X = np.asarray(X)
        return np.array([
            self.predict_one(x, self.tree)
            for x in X
        ])


# Example
X = np.array([
    [2, 3],
    [1, 2],
    [3, 1],
    [8, 7],
    [9, 8],
    [10, 9]
])

y = np.array([0, 0, 0, 1, 1, 1])

model = DecisionTree(max_depth=3)
model.fit(X, y)

predictions = model.predict(X)

print("Predictions:", predictions)


Predictions: [0 0 0 1 1 1]
